# MirrorTopology Step 1 — Phase B・B-3-2 B：登録規模 共有 W₂ null asset（v0.5・無効世代の非公開・complete 照合・backup 状態の分離）
等方 null pool（w2_isotropic・N=6×10⁵・6000 cluster・両 selection）を校正 bank（N=2×10⁵；A10 official と照合）の whitening で変換し，exact POT W₂ で n_sub 2000／5000 × B_max=1000 × 3 pair の null 列を一度だけ生成する。**所要時間 2.5〜4 h**。
- ローカル checkpoint：builder が 25 replicate ごとに atomic 保存（digest 束縛 envelope）。
- 永続保存：`PERSIST_DIR`（Google Drive）へ **60 s ごとに新しい世代として publish**（コピー後に SHA と envelope を再検証してから generation manifest と LATEST を切替；直前世代は保持；失敗は記録され旧世代を壊さない）。
- publisher は**単一 worker**（60 s 周期＋script 終了後の最終 publish を同じ worker が実行；worker の終了を確認してからログ close と出力確定）。publish は directory lock で直列化。
- snapshot 契約：期待名↔identity.n_sub・done↔payload 件数・共通 identity・逐次 stage（2000 が完了していない 5000 は再開不可）を保存／復元の入口で検査し，generation manifest は**コピー後の envelope**から作る。
- helper（`ckpt_persist`）は NumPy 非依存；環境 lock 後に import。
- 再開不能な snapshot（先行 stage 欠落・identity 不一致）は **publish されず LATEST・保持世代を消費しない**（helper が公開前とコピー後に拒否；復元側も拒否）。
- `.publish.lock`（永続先の directory lock）が残存した場合：旧 publisher の停止を確認してから手動で削除するか，新しい `PERSIST_DIR` を保存先にする（無条件削除はしない）。
- 最終 record は **数値 script の成功**と **persistent backup の状態**（publish 回数・失敗数・最終世代）を分けて記録する；`B3_2B_PASS` は前者＋worker 終了を表し，Drive 保存成功の証明ではない。
- `MODE='fresh'`：新 run。`MODE='resume'`：`RESUME_SOURCE` に (a) ローカル run directory または (b) Drive の永続 run root を指定。lock（commit／inventory／launcher／pins）を**何も触る前に**照合し，Drive からの復元は**新しい空の staging run**へ行う（既存 run への merge はしない）。期待 checkpoint（`null_nsub2000.json`／`null_nsub5000.json`）が無ければ停止（無関係な JSON は再開根拠にしない；2000 のみの partial は正当）。過去の run manifest は attempt 別に保持。label なし。

In [ ]:
# --- 0. OUTER LAUNCHER LOCK (the only editable cell)
REPO_URL = 'https://github.com/tsujikeita/mirror-topology.git'
REPO_COMMIT = '<full 40-hex commit of the verification target>'
EXPECTED_INVENTORY_SHA256 = '<sha256 of engine/phaseB/B2_completion_inventory.json inside that commit>'
LAUNCHER_ID = 'MirrorTopology_Step1_B3_2B_shared_null_v0.5'
MODE = 'fresh'                 # 'fresh' | 'resume'
RESUME_SOURCE = ''             # resume only: a local run dir (/content/b3_2B_runs/<id>) OR a persistent run root (PERSIST_DIR/<id>)
PERSIST_DIR = '/content/drive/MyDrive/mirror_topology_b3_2B'   # persistent root for checkpoint generations ('' disables persistence)


In [ ]:
# --- 1. verify the launcher against the source BEFORE any copy; fresh run or verified resume; scratch checkout; inventory-bound pins/script
import subprocess, sys, os, json, hashlib, shutil, time, re
sha=lambda p: hashlib.sha256(open(p,'rb').read()).hexdigest()
assert re.fullmatch(r'[0-9a-f]{40}', REPO_COMMIT) and re.fullmatch(r'[0-9a-f]{64}', EXPECTED_INVENTORY_SHA256), 'launcher lock not filled'
assert MODE in ('fresh','resume'), MODE
if PERSIST_DIR:
    from google.colab import drive; drive.mount('/content/drive', force_remount=False); os.makedirs(PERSIST_DIR, exist_ok=True)
# scratch checkout first (needed to load pins for the lock check)
STAMP=time.strftime('%Y%m%dT%H%M%SZ', time.gmtime()); SCRATCH=f'/content/b3_2B_scratch/{STAMP}'; os.makedirs(SCRATCH)
subprocess.run(['git','clone','-q',REPO_URL,SCRATCH],check=True); subprocess.run(['git','-C',SCRATCH,'checkout','-q',REPO_COMMIT],check=True)
head=subprocess.check_output(['git','-C',SCRATCH,'rev-parse','HEAD']).decode().strip(); assert head==REPO_COMMIT, head
assert subprocess.check_output(['git','-C',SCRATCH,'status','--porcelain']).decode().strip()=='', 'scratch tree not clean'
MT=SCRATCH; PHASEB=f'{MT}/engine/phaseB'; INV=f'{PHASEB}/B2_completion_inventory.json'; inv_sha=sha(INV); assert inv_sha==EXPECTED_INVENTORY_SHA256, inv_sha
inv=json.load(open(INV)); PINS=f'{PHASEB}/b3/b3_2_pins.json'; SCRIPT=f'{PHASEB}/b3/b3_2_shared_null.py'
assert sha(PINS)==inv['b3_sha256']['b3/b3_2_pins.json'] and sha(SCRIPT)==inv['b3_sha256']['b3/b3_2_shared_null.py']
pins=json.load(open(PINS)); assert 'repo' not in pins and pins['engine_version']==inv['engine_version'] and pins['schema']=='b3_2_pins_v1'
lock=dict(launcher_id=LAUNCHER_ID, repo_url=REPO_URL, commit=head, inventory_sha256=inv_sha, pins_sha256=sha(PINS), engine_version=inv['engine_version'], mode=MODE, persist_dir=PERSIST_DIR)
def same_lock(a,b): return all(a.get(k)==b.get(k) for k in ('commit','inventory_sha256','launcher_id','pins_sha256'))
print('launcher lock prepared (resume/restore happens after the environment lock):', lock)


In [ ]:
# --- 2. environment lock
ex=pins['environment']
subprocess.run(['pip','install','-q',f"numpy=={ex['numpy']}",f"scipy=={ex['scipy']}",f"healpy=={ex['healpy']}",f"pot=={ex['pot']}",f"camb=={ex['camb']}",'threadpoolctl'],check=True)
os.environ['OPENBLAS_NUM_THREADS']='2'; os.environ['OMP_NUM_THREADS']='2'
import platform, numpy, scipy, healpy, ot, camb
live=dict(python=platform.python_version(), numpy=numpy.__version__, scipy=scipy.__version__, healpy=healpy.__version__, pot=ot.__version__, camb=camb.__version__)
mism={k:(live[k],ex[k]) for k in live if live[k]!=ex[k]}; assert not mism, f'environment lock failed: {mism}'; print('environment lock OK', live)


In [ ]:
# --- 2b. fresh run or verified resume/restore (helper imported only AFTER the environment lock; the helper itself is NumPy-free)
sys.path.insert(0, PHASEB); from step1_engine.ckpt_persist import inspect_local, restore_generation, latest_generation, verify_generation
if MODE=='fresh':
    assert not RESUME_SOURCE, 'fresh mode must not name a resume source'
    RUN=f'/content/b3_2B_runs/{STAMP}'; OUT=f'{RUN}/out'; os.makedirs(f'{OUT}/null/ckpt'); lock['run_dir']=RUN; lock['run_id']=STAMP
    json.dump(lock, open(f'{OUT}/launcher_lock.json','w'), indent=1); json.dump([dict(kind='fresh', at=STAMP)], open(f'{OUT}/attempts.json','w'), indent=1)
else:
    src=RESUME_SOURCE.rstrip('/'); assert src and os.path.isdir(src), f'resume source not found: {src}'
    if os.path.exists(f'{src}/out/launcher_lock.json'):                                                    # (a) local run: verify its lock and snapshot state, use it in place (no copy)
        prev=json.load(open(f'{src}/out/launcher_lock.json')); assert same_lock(prev, lock), f'previous lock differs from the current launcher: {prev}'
        info=inspect_local(f'{src}/out/null/ckpt'); assert info['resumable'], f'local run is not a valid resume state: {info["stage_state"]["reason"]} (unrelated: {info["unrelated"]})'
        RUN=src; OUT=f'{RUN}/out'; lock['run_dir']=RUN; lock['run_id']=prev.get('run_id', os.path.basename(src)); restored=None
    else:                                                                                                  # (b) persistent run root: verify generation + lock BEFORE any copy, restore into a NEW empty staging run
        gen=latest_generation(src); assert gen, 'no complete generation in the persistent source'
        man=verify_generation(gen); assert same_lock(man['lock'], lock), f'generation lock differs from the current launcher: {man["lock"]}'
        assert man.get('resumable') is True, f'latest generation is not a valid resume state: {man.get("stage_reason")}'
        RUN=f'/content/b3_2B_runs/{STAMP}_restored_from_{os.path.basename(src)}'; OUT=f'{RUN}/out'; os.makedirs(f'{OUT}/null'); restored=restore_generation(src, f'{OUT}/null/ckpt', lock)
        assert restored['resumable'], restored; lock['run_dir']=RUN; lock['run_id']=man['lock'].get('run_id', os.path.basename(src)); json.dump(lock, open(f'{OUT}/launcher_lock.json','w'), indent=1); print('restored', restored)
    hist=f'{OUT}/attempts.json'; H=json.load(open(hist)) if os.path.exists(hist) else []
    if os.path.exists(f'{OUT}/null/b3_2_shared_null_run_manifest.json'):
        adir=f'{OUT}/attempts/{len(H):03d}'; os.makedirs(adir, exist_ok=True); shutil.copy2(f'{OUT}/null/b3_2_shared_null_run_manifest.json', f'{adir}/b3_2_shared_null_run_manifest.json')
    H.append(dict(kind='resume', at=STAMP, source=src, restored=restored, state=inspect_local(f'{OUT}/null/ckpt')['stage_state'])); json.dump(H, open(hist,'w'), indent=1)
PERSIST_RUN=f"{PERSIST_DIR}/{lock['run_id']}" if PERSIST_DIR else None; print(lock, 'persist:', PERSIST_RUN)


In [ ]:
# --- 3. shared null (resumable). ONE publisher worker: periodic publish every 60 s while the script runs, then the FINAL publish by the same worker after the script exits;
#        the cell proceeds to logging close / output inventory only after the worker has terminated (join without timeout; a slow copy completes or fails, never overlaps).
NO=f'{OUT}/null'; os.makedirs(f'{NO}/ckpt', exist_ok=True)
import threading, traceback
from step1_engine.ckpt_persist import publish_generation
plog=open(f'{OUT}/persist_log.txt','a'); plog_lock=threading.Lock()
def log_persist(msg):
    with plog_lock: plog.write(f'{time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())} {msg}\n'); plog.flush()
state={'script_done': threading.Event(), 'publishes': 0, 'failures': 0, 'last': None}
def publish(reason):
    if not PERSIST_RUN: return
    try:
        g=publish_generation(f'{NO}/ckpt', PERSIST_RUN, lock, note=lambda *m: log_persist(' '.join(map(str,m)))); state['publishes']+=1; state['last']=g; log_persist(f'{reason}: published {g}')
    except Exception as ex: state['failures']+=1; log_persist(f'{reason}: publish FAILED {ex!r} (previous generation kept)')
def worker():
    try:
        while not state['script_done'].wait(60): publish('periodic')
        publish('final')                                                                                   # the single publisher performs the final publish itself
    except BaseException as ex: log_persist(f'worker terminated by {ex!r}\n{traceback.format_exc()}')
th=threading.Thread(target=worker, name='publisher', daemon=False); th.start()
rc=None
try:
    rc=subprocess.run([sys.executable,SCRIPT,'--mt',MT,'--phaseb',PHASEB,'--out',NO,'--profile','assets_official'],capture_output=True,text=True)
finally:
    state['script_done'].set(); t_join=time.time(); th.join()                                                # wait for the worker's final publish (no timeout)
    log_persist(f'worker joined after {time.time()-t_join:.1f}s; publishes={state["publishes"]} failures={state["failures"]} last={state["last"]}'); plog.close()
open(f'{OUT}/launcher_script_stdout.txt','a').write(rc.stdout if rc else ''); open(f'{OUT}/launcher_script_stderr.txt','a').write(rc.stderr if rc else ''); print((rc.stdout if rc else '')[-3000:])
rm=json.load(open(f'{NO}/b3_2_shared_null_run_manifest.json')); script_ok=bool(rc is not None and rc.returncode==0 and rm.get('B3_2B_PASS') is True); print('script rc', rc.returncode if rc else None, 'B3_2B_PASS', rm.get('B3_2B_PASS'), rm.get('failures'), 'publisher alive:', th.is_alive())


In [ ]:
# --- final record (composed; excludes itself) + zip for the audit
def inventory(root, exclude=()):
    out={}
    for d,_,fs in os.walk(root):
        for f in fs:
            p=os.path.join(d,f); rel=os.path.relpath(p, root)
            if rel in exclude: continue                                   # checkpoints (digest-bound envelopes) are part of the resume evidence and are listed
            out[rel]=dict(sha256=sha(p), bytes=os.path.getsize(p))
    return out
assert not th.is_alive(), 'publisher worker still alive: final record must not be written'
final=dict(launcher=lock, B3_2B_PASS=bool(script_ok and not th.is_alive()), stages=dict(checkout=True, environment=live, script_returncode=rc.returncode, script_pass=rm.get('B3_2B_PASS'), script_gates=rm.get('gates'), script_failures=rm.get('failures'), timings=rm.get('timings'), publisher=dict(publishes=state['publishes'], failures=state['failures'], last_generation=state['last'], joined=True)), backup_state=dict(persistent_backup_ok=bool(PERSIST_RUN and state['failures']==0 and state['last'] is not None), scope='checkpoint generations + generation manifests/lock only (not the full RUN/out); B3_2B_PASS does not certify the backup'))
final['output_inventory']=inventory(OUT, exclude=('b3_2B_final_record.json','b3_2B_return_list.json'))
json.dump(final, open(f'{OUT}/b3_2B_final_record.json','w'), indent=1); json.dump(dict(final_record_sha256=sha(f'{OUT}/b3_2B_final_record.json'), files=final['output_inventory']), open(f'{OUT}/b3_2B_return_list.json','w'), indent=1)
print(json.dumps({k:final[k] for k in ('B3_2B_PASS',)}, indent=1), 'run dir:', RUN)
import shutil
from google.colab import files
p = shutil.make_archive(f'/content/b3_2B_out_{REPO_COMMIT[:12]}', 'zip', root_dir=OUT); print(p, os.path.getsize(p)); files.download(p)
